# German frame-classification head — `train_frame2_de`

Trains on the **SALSA** German corpus with the **`deepset/gbert-large`** backbone.

### One-time setup (on your own machine)
The SALSA corpus is licence-restricted, so it can't live in a public repo — you upload it (plus the code) to *your* Drive once, as a single zip:

```bash
cd ~/Desktop/Academic_projects/Texture_Frames
zip -r texture_frames_colab.zip \
    encoder_parser \
    German_parser/*.py \
    German_parser/extracted/salsa_release.xml \
    German_parser/extracted/salsa_frames.xml
```
Then upload `texture_frames_colab.zip` to the **top level of your Google Drive** (`MyDrive/`).

### Runtime
`Runtime → Change runtime type → GPU`. **A100** fits `batch_size=16`; on **L4/T4** use `batch_size=8`.

### Crash safety
Checkpoints are written to **your Drive** every epoch. **If Colab disconnects, just re-run the Train cell** — it auto-resumes from the last epoch on Drive.

In [ ]:
!nvidia-smi

## 1. Mount Drive & unpack the project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
ZIP = "/content/drive/MyDrive/texture_frames_colab.zip"
assert os.path.exists(ZIP), f"Upload the project zip to {ZIP} first (see the intro cell)."
!rm -rf /content/Texture_Frames && mkdir -p /content/Texture_Frames
!unzip -q "$ZIP" -d /content/Texture_Frames

base = "/content/Texture_Frames"
need = ["German_parser/salsa_loader.py", "German_parser/salsa_lexicon.py",
        "encoder_parser/model_frame2.py", "encoder_parser/model_args2.py",
        "German_parser/extracted/salsa_release.xml",
        "German_parser/extracted/salsa_frames.xml"]
missing = [p for p in need if not os.path.exists(os.path.join(base, p))]
print("MISSING:" , missing or "none — all code + data present")
assert not missing, "Re-make the zip; some files are missing."

In [ ]:
!pip install -q "transformers==4.57.6" "huggingface_hub<1.0" "accelerate>=0.30" sentencepiece simplemma "numpy>=2"

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")
sys.path.insert(0, "/content/Texture_Frames/German_parser")
os.chdir("/content/Texture_Frames/German_parser")

import numpy, torch, transformers
print("numpy", numpy.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — set Runtime → GPU."
from huggingface_hub import __version__ as _hv
assert int(_hv.split(".")[0]) < 1, (
    f"huggingface_hub {_hv} is loaded, but transformers 4.57.6 needs <1.0 \u2014 "
    "do Runtime \u2192 Restart session, then Run all from the top.")

## 2. Sanity check — data loads and markers land on the trigger

In [ ]:
from salsa_loader import load_frame_examples
from salsa_lexicon import SalsaLexicon
from salsa_frame_data import mark_trigger

ex = load_frame_examples("dev", drop_unannotated=True)
lx = SalsaLexicon()
print("dev examples:", len(ex), "| frame vocabulary:", len(lx.frame2id()))
text, loc, frame = ex[0]
print("gold frame :", frame)
print("marked     :", mark_trigger(text, loc)[:120])

## 3. Train  →  checkpoints to Drive (resumable)

~20–40 min on A100. Re-run this cell after any disconnect to resume.

In [ ]:
import train_frame2_de

CKPT_DIR = "/content/drive/MyDrive/Texture_Frames/checkpoints/frame2_de"
print("checkpointing to Drive:", CKPT_DIR)

model, tokenizer, lexicon = train_frame2_de.train(
    base_model="deepset/gbert-large",
    output_dir=CKPT_DIR,       # on Drive → survives disconnects
    epochs=5,
    batch_size=16,             # L4/T4: set to 8
    lr=1e-5,
    max_length=320,
    resume=True,               # auto-continue from the last Drive checkpoint
)

## 4. Evaluate — candidate-mask bias sweep (pick on dev, report on test)

In [ ]:
from train_frame2_de import evaluate_frame2_de, print_report

print("### DEV — pick the winning bias ###")
print_report(evaluate_frame2_de(model, tokenizer, lexicon, split="dev"))

print("\n### TEST — report the dev-chosen bias ###")
print_report(evaluate_frame2_de(model, tokenizer, lexicon, split="test"))

## 5. Save the final model to Drive

In [ ]:
import os, shutil
MODEL_DIR = "/content/drive/MyDrive/Texture_Frames/models/frame2_de"
os.makedirs(MODEL_DIR, exist_ok=True)

for f in ["frame2_model.pt", "frame2id.json"]:
    shutil.copy(os.path.join(CKPT_DIR, f), MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("saved to:", MODEL_DIR)
for f in sorted(os.listdir(MODEL_DIR)):
    print(f"  {f:28} {os.path.getsize(os.path.join(MODEL_DIR, f))/1e6:8.1f} MB")

## Notes

- **Disconnected?** Re-run cells 1 → 3. The Train cell resumes from the last checkpoint on Drive.
- **Reclaim space:** after the final model is saved (cell 5), you may delete `MyDrive/Texture_Frames/checkpoints/frame2_de/` — keep `models/frame2_de/`.
- The **trigger head needs no training** — it's a lexicon rule (`salsa_trigger.py`, F1 ≈ 0.88). Only the frame and argument heads are trained.